<a href="https://colab.research.google.com/github/hareezzvijey/Amrita_AI_Research_Internship/blob/main/Day_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Exp1: Neural Network from scratch

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
def log_loss(y_true, y_predicted):
  # Binary cross entropy
  epsilon = 1e-15
  y_predicted_new = [max(i, epsilon) for i in y_predicted]
  y_predicted_new = [min(i, 1-epsilon) for i in y_predicted_new]
  y_predicted_new = np.array(y_predicted_new) # Puts value btw (ϵ,1−ϵ) - 0.000000000000001  to  0.999999999999999
  return -np.mean(y_true*np.log(y_predicted_new)+(1-y_true)*np.log(1-y_predicted_new))

def sigmoid_numpy(x):
  return 1/(1+np.exp(-x))

def gradient_descent(age, affordability,y_true ,epochs, loss_threshold):
  w1 = w2 = 1
  bias = 0
  learning_rate = 0.5
  n = len(age)

  for i in range(epochs):
    weighted_sum = w1*age + w2*affordability + bias
    y_predicted = sigmoid_numpy(weighted_sum)

    loss = log_loss(y_true, y_predicted)

    # How much does the parameters contribute to the error = x1*error
    w1d = (1/n)*(np.dot(np.transpose(age),(y_predicted-y_true)))
    w2d = (1/n)*(np.dot(np.transpose(affordability),(y_predicted-y_true)))
    bias_d = np.mean(y_predicted-y_true)

    w1 = w1 - learning_rate*w1d
    w2 = w2 - learning_rate*w2d
    bias = bias - learning_rate*bias_d

    if loss<=loss_threshold:
      break

    print(f'Epoch:{i}, w1:{w1}, w2:{w2}, bias:{bias}, loss:{loss}')

  return w1, w2, bias

def predict(x_test):
    weighted_sum = w1 * x_test['age'] + w2 * x_test['affordability'] + bias
    return sigmoid_numpy(weighted_sum)

In [ ]:
df = pd.read_csv("insurance_data.csv")
df.columns = ['age', 'affordability', 'insurance']

In [ ]:
from sklearn.model_selection import train_test_split

X = df[['age', 'affordability']]
y = df['insurance']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=25)

In [ ]:
X_train_scaled = X_train.copy()
X_train_scaled['age'] = X_train_scaled['age']/100

X_test_scaled = X_test.copy()
X_test_scaled['age'] = X_test_scaled['age']/100

w1, w2, bias = gradient_descent(
    X_train_scaled['age'],
    X_train_scaled['affordability'],
    y_train,
    5000,
    0.40
)

prediction = predict(X_test_scaled)

Epoch:0, w1:1.0137421856372864, w2:1.0018194535344567, bias:0.013057234829616175, loss:0.5474877950947307
Epoch:1, w1:1.0266990642359775, w2:1.0028401128302602, bias:0.024228734533580626, loss:0.5467994150456048
Epoch:2, w1:1.0389775851604188, w2:1.0031857379916518, bias:0.03377213951913323, loss:0.5462396652596263
Epoch:3, w1:1.0506684797009607, w2:1.002960300479936, bias:0.04190645123057056, loss:0.5457766308852822
Epoch:4, w1:1.0618490368867655, w2:1.002251575470354, bias:0.04881855675216937, loss:0.5453866545855072
Epoch:5, w1:1.072585333875309, w2:1.0011339816124223, bias:0.05466849047270809, loss:0.5450521144439191
Epoch:6, w1:1.0829340430278507, w2:0.9996708472227556, bias:0.059593711388316394, loss:0.5447598276646058
Epoch:7, w1:1.0929439073649196, w2:0.9979162348699814, bias:0.06371260620376228, loss:0.5444998946296964
Epoch:8, w1:1.1026569539011402, w2:0.9959164226052117, bias:0.06712737821322623, loss:0.5442648558516248
Epoch:9, w1:1.11210949801979, w2:0.9937111157184262, bi

In [ ]:
print(round(prediction,0))

2     1.0
10    1.0
21    1.0
11    1.0
14    0.0
9     1.0
dtype: float64


In [ ]:
print(y_test)

2     0
10    1
21    0
11    0
14    0
9     1
Name: insurance, dtype: int64


### Exp2: Implementation of the stochastic gradient descent

In [ ]:
def stochastic_gradient_descent(age, affordability, y_true ,epochs, loss_threshold):
  w1 = w2 = 1
  bias = 0
  learning_rate = 0.01
  n = len(age)

  for i in range(epochs):
      for j in range(len(age)):
          weighted_sum = w1*age.iloc[j] + w2*affordability.iloc[j] + bias
          y_pred = sigmoid_numpy(weighted_sum)

          error = y_pred - y_true.iloc[j]

          w1 -= learning_rate * error * age.iloc[j]
          w2 -= learning_rate * error * affordability.iloc[j]
          bias -= learning_rate * error

      # After one full pass
      y_predicted_epoch = sigmoid_numpy(w1*age + w2*affordability + bias)
      loss = log_loss(y_true, y_predicted_epoch)

      if loss<=loss_threshold:
          break

      print(f'Epoch:{i}, w1:{w1}, w2:{w2}, bias:{bias}, loss:{loss}')

  return w1, w2, bias

In [ ]:
X_train_scaled = X_train.copy()
X_train_scaled['age'] = X_train_scaled['age']/100

X_test_scaled = X_test.copy()
X_test_scaled['age'] = X_test_scaled['age']/100

w1, w2, bias = stochastic_gradient_descent(
    X_train_scaled['age'],
    X_train_scaled['affordability'],
    y_train,
    5000,
    0.40
)

prediction = predict(X_test_scaled)

Epoch:0, w1:1.3979123979693235, w2:1.2240383543496418, bias:0.3729387791937168, loss:0.5611690753732922
Epoch:1, w1:1.6155662515038576, w2:1.213897277524136, bias:0.3110275635949872, loss:0.559360467989852
Epoch:2, w1:1.8182736921296492, w2:1.194997926095146, bias:0.23441232453268912, loss:0.55588223998003
Epoch:3, w1:2.0131452940739845, w2:1.1763392351709763, bias:0.1600990178661016, loss:0.5525745782138491
Epoch:4, w1:2.200815378942967, w2:1.1584097376966507, bias:0.08877507319743871, loss:0.5495118130784706
Epoch:5, w1:2.3815920055994857, w2:1.141241765777182, bias:0.020328858010736528, loss:0.5466796367139343
Epoch:6, w1:2.5557603353834093, w2:1.124825280918063, bias:-0.045380113860339355, loss:0.5440603951485653
Epoch:7, w1:2.72359659153683, w2:1.1091369221747565, bias:-0.10848474198201102, loss:0.5416374066671521
Epoch:8, w1:2.885367906181941, w2:1.0941466682466128, bias:-0.16911003439699773, loss:0.5393951802321916
Epoch:9, w1:3.041331768931488, w2:1.0798214308665717, bias:-0.22

In [ ]:
print(round(prediction,0))

2     1.0
10    1.0
21    1.0
11    1.0
14    0.0
9     1.0
dtype: float64


In [ ]:
print(y_test)

2     0
10    1
21    0
11    0
14    0
9     1
Name: insurance, dtype: int64


### Exp3: Mini Batch Gradient Descent

In [ ]:
def stochastic_gradient_descent(age, affordability,y_true ,epochs, loss_threshold):
  w1 = w2 = 1
  bias = 0
  learning_rate = 0.5
  n = len(age)

  batch_size = 5

  for i in range(epochs):
      for j in range(0, len(age), batch_size):
          age_batch = age[j:j+batch_size]
          aff_batch = affordability[j:j+batch_size]
          y_batch = y_true[j:j+batch_size]

          weighted_sum = w1*age_batch + w2*aff_batch + bias
          y_pred = sigmoid_numpy(weighted_sum)

          error = y_pred - y_batch

          w1 -= learning_rate * np.dot(age_batch, error)/len(age_batch)
          w2 -= learning_rate * np.dot(aff_batch, error)/len(age_batch)
          bias -= learning_rate * np.mean(error)

      y_predicted_epoch = sigmoid_numpy(w1*age + w2*affordability + bias)
      loss = log_loss(y_true, y_predicted_epoch)

      if loss<=loss_threshold:
          break

      print(f'Epoch:{i}, w1:{w1}, w2:{w2}, bias:{bias}, loss:{loss}')

  return w1, w2, bias

In [ ]:
X_train_scaled = X_train.copy()
X_train_scaled['age'] = X_train_scaled['age']/100

X_test_scaled = X_test.copy()
X_test_scaled['age'] = X_test_scaled['age']/100

w1, w2, bias = stochastic_gradient_descent(
    X_train_scaled['age'],
    X_train_scaled['affordability'],
    y_train,
    5000,
    0.40
)

prediction = predict(X_test_scaled)

Epoch:0, w1:1.0622809380053617, w2:1.0276110730295436, bias:0.05029366660678905, loss:0.5451088183708731
Epoch:1, w1:1.1110283114867223, w2:1.0400363618036512, bias:0.06790443505991539, loss:0.5441654559882548
Epoch:2, w1:1.1531523952322038, w2:1.0452372251512958, bias:0.06989100466778672, loss:0.5435230206043106
Epoch:3, w1:1.1918826260945032, w2:1.0468489699560155, bias:0.06425295494728606, loss:0.5429397641342625
Epoch:4, w1:1.228788424082807, w2:1.046622284507597, bias:0.0548823199229273, loss:0.5423669337336438
Epoch:5, w1:1.264648303506837, w2:1.0454258713349813, bias:0.04370783975162201, loss:0.541797368764691
Epoch:6, w1:1.299852916565405, w2:1.0436993861692003, bias:0.031693575950200545, loss:0.5412317728873937
Epoch:7, w1:1.33459982328124, w2:1.041669301559862, bias:0.019322785020560646, loss:0.5406715865414082
Epoch:8, w1:1.3689896966580237, w2:1.0394548123905276, bias:0.006837350575734341, loss:0.5401178025151782
Epoch:9, w1:1.4030743929815475, w2:1.0371206152637633, bias:-

In [ ]:
print(round(prediction,0))

2     1.0
10    1.0
21    1.0
11    1.0
14    0.0
9     1.0
dtype: float64


In [ ]:
print(y_test)

2     0
10    1
21    0
11    0
14    0
9     1
Name: insurance, dtype: int64


### Tensorflow using Adam optimizer and l2 regularization

In [ ]:
import tensorflow as tf
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras import regularizers

In [ ]:
df = pd.read_csv("insurance_data.csv")

X = df[['Age', 'Affordability']]
y = df['Bought_insurance']

scaler = MinMaxScaler()
X['Age'] = scaler.fit_transform(X[['Age']])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

/tmp/ipykernel_503/3725732093.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Age'] = scaler.fit_transform(X[['Age']])


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(
        1,
        input_shape=(2,),
        activation='sigmoid',
        kernel_regularizer=tf.keras.regularizers.l2(0.01)
    )
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=5,
    verbose=1
)

Epoch 1/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.4091 - loss: 1.0387  
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4091 - loss: 1.0351 
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4091 - loss: 1.0322     
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4091 - loss: 1.0290 
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4091 - loss: 1.0261 
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4091 - loss: 1.0233 
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4545 - loss: 1.0200 
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4545 - loss: 1.0173 
Epoch 9/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4545 - loss: 1.0150 
Epoch 10/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4545 - loss: 1.0120 
Epoch 11/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4545 - loss: 1.0093 
Epoch 12/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4545 

In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print("Accuracy:", acc)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step - accuracy: 0.3333 - loss: 0.9070
Accuracy: 0.3333333432674408


### Exp5: Backpropagation implementation

In [ ]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(output):
    return output * (1 - output)

def binary_loss(actual, predicted):
    epsilon = 1e-15
    predicted = np.clip(predicted, epsilon, 1 - epsilon)
    return -np.mean(actual*np.log(predicted) + (1-actual)*np.log(1-predicted))

def train_model(X, y, epochs=1000, lr=0.01):

    weights = np.random.randn(X.shape[1])
    bias = 0

    n = len(y)

    for i in range(epochs):
        linear_output = np.dot(X, weights) + bias
        predicted_output = sigmoid(linear_output)

        loss = binary_loss(y, predicted_output)


        error = predicted_output - y
        slope = sigmoid_derivative(predicted_output)

        delta = error * slope

        weight_gradient = np.dot(X.T, delta) / n
        bias_gradient = np.mean(delta)

        weights -= lr * weight_gradient
        bias -= lr * bias_gradient

        if i % 100 == 0:
            print(f"Epoch {i}, Loss: {loss}")

    return weights, bias

In [ ]:
def predict(X, weights, bias):
    linear_output = np.dot(X, weights) + bias
    predicted_output = sigmoid(linear_output)
    return (predicted_output > 0.5).astype(int)

In [ ]:
weight, bias = train_model(X_train, y_train)

Epoch 0, Loss: 0.6562495305000986
Epoch 100, Loss: 0.6530273562603637
Epoch 200, Loss: 0.6501739540801442
Epoch 300, Loss: 0.6476284787387224
Epoch 400, Loss: 0.6453405056385471
Epoch 500, Loss: 0.6432682989867264
Epoch 600, Loss: 0.6413773301797521
Epoch 700, Loss: 0.6396390282571404
Epoch 800, Loss: 0.6380297365169714
Epoch 900, Loss: 0.6365298472505541


In [ ]:
prediction = predict(X_test, weight, bias)
print(prediction)

[1 1 1 1 1 1]


In [ ]:
print(y_test)

9     1
25    1
8     1
21    0
0     1
12    1
Name: Bought_insurance, dtype: int64
